# Multi-tenancy as a measured property

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/11-tenancy/tenancy.ipynb)

Built from [`cookbook/book/chapters/11-tenancy/tenancy.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/11-tenancy/tenancy.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `set_tenant` · `list_sources` · `sql` (the analyzer's discriminator rewrite)
· **Theory:** tenant isolation as a property of the catalog + the query analyzer, not a
feature flag · **Rail:** tenancy (`rails.tenant` / `assert_listing_isolated` /
`assert_rows_isolated`) + measurement.

The Air Routes on-ramp introduced the engine's *true* tenancy contract. This chapter does not
restate it as prose — it **measures** it, live over the ogbn-arxiv papers, as three
isolation properties plus one consequence. Every number is a verdict: a hard zero where
isolation must hold, a positive count where the honest caveat says data is visible.

The model has exactly two genuine isolation layers and one honest caveat:

1. **catalog-listing isolation** — `list_sources` filters the registry to
   `tenant_id = $cur OR IS NULL`; tenant A's listing excludes B's registration.
2. **row-level discriminator-column isolation** — the analyzer injects
   `tenant_id = $cur OR IS NULL` onto a `TableScan` *only when* the queried table carries
   a `tenant_id` column, so the SAME tagged source returns disjoint rows under A vs B.
3. **the caveat** — a discriminator-**less** source is globally readable: A sees ALL of
   B's rows when it names the source. The engine does not authenticate; access-gating
   lives above it.

The two tenants are an honest, disjoint partition of the papers: every paper
belongs to exactly one tenant by its subject class. The tenant ids are opaque UUIDs — the
engine validates the form, never who the tenant is.

## The setup — two tenants over one corpus

We reuse `jammi_cookbook.rails` verbatim — `tenant` (the bind-in-place/restore context
manager), `assert_listing_isolated`, and `assert_rows_isolated`, the corrected helpers,
not a re-implementation. `set_tenant` binds the scope to the connection *in place*; the
bound scope drives both isolation layers (`tenant_scope` is the block-scoped form the
`tenant` helper delegates to).

In [ ]:
import tempfile

import jammi
import pyarrow as pa
import pyarrow.parquet as pq
from jammi_cookbook import contracts, datasets, encoders, scale
from jammi_cookbook.rails import assert_listing_isolated, assert_rows_isolated, tenant

SCALE = scale.current()
TENANT_A = "aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa"
TENANT_B = "bbbbbbbb-bbbb-bbbb-bbbb-bbbbbbbbbbbb"

work = tempfile.mkdtemp()
db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)
papers = db.sql(
    f"SELECT paper_id, title, abstract, subject FROM {arxiv.papers}.public.{arxiv.papers}"
).to_pylist()
subject = {p["paper_id"]: p["subject"] for p in papers}
subjects = sorted({p["subject"] for p in papers})
a_subjects = set(subjects[::2])
a_papers = sorted(pid for pid in subject if subject[pid] in a_subjects)
b_papers = sorted(pid for pid in subject if subject[pid] not in a_subjects)


def write_src(name, table):
    path = f"{work}/{name}.parquet"
    pq.write_table(table, path)
    db.add_source(name, url=path, format="parquet")


print(f"partition: A {len(a_papers)} / B {len(b_papers)} papers (disjoint, tiling the corpus)")

## Property 1 — catalog-listing isolation (a hard zero)

Each tenant registers its own source under its own scope. Under tenant A, `list_sources`
must not surface B's registration.

In [ ]:
with tenant(db, TENANT_A):
    write_src("papers_a", pa.table({"paper_id": a_papers}))
with tenant(db, TENANT_B):
    write_src("papers_b", pa.table({"paper_id": b_papers}))

with tenant(db, TENANT_A):
    a_listed = [s["source_id"] for s in db.list_sources()]

assert_listing_isolated(a_listed, {"papers_b"}, tenant_id=TENANT_A)
listing_leak = len(set(a_listed) & {"papers_b"})
print(f"tenant A lists: {sorted(a_listed)}")
print(f"B's 'papers_b' leaked into A's listing: {listing_leak}")

## Property 2 — discriminator-column row isolation (a hard zero)

Now the load-bearing layer. We register **one shared source** as a global
(`tenant_id IS NULL`), carrying a `tenant_id` discriminator column that tags each row to a
tenant. When tenant A reads it, the analyzer injects `tenant_id = $cur OR IS NULL` onto the
scan, so A sees only its own rows — none of B's.

In [ ]:
db.set_tenant("")  # register the shared tagged source as a global
write_src("papers_tagged", pa.table({
    "paper_id": [p["paper_id"] for p in papers],
    "title": [p["title"] for p in papers],
    "abstract": [p["abstract"] for p in papers],
    "subject": [p["subject"] for p in papers],
    "tenant_id": [TENANT_A if p["subject"] in a_subjects else TENANT_B for p in papers],
}))

with tenant(db, TENANT_A):
    seen_a = [r["paper_id"] for r in db.sql(
        "SELECT paper_id FROM papers_tagged.public.papers_tagged").to_pylist()]

assert_rows_isolated(seen_a, set(b_papers), tenant_id=TENANT_A)
discriminator_leak = len(set(seen_a) & set(b_papers))
print(f"tenant A reads the shared tagged source: {len(seen_a)} rows")
print(f"B-tagged rows that leaked to A: {discriminator_leak}")

This is the isolation layer that actually scopes *data*: one physical source, disjoint
reads, enforced by the analyzer because the discriminator column exists.

## Property 3 — the caveat: a discriminator-less source is globally readable

The honest limit, asserted as a *positive* count, not hidden. A source with **no**
`tenant_id` column gets no scan rewrite — it is globally readable by any tenant that names
it. We register B's papers under B with no discriminator column, then read it under A: A
sees **all** of B's rows.

In [ ]:
with tenant(db, TENANT_B):
    write_src("papers_b_nodisc", pa.table({"paper_id": b_papers}))

with tenant(db, TENANT_A):
    seen_global = [r["paper_id"] for r in db.sql(
        "SELECT paper_id FROM papers_b_nodisc.public.papers_b_nodisc").to_pylist()]

caveat_visible = len(set(seen_global) & set(b_papers))
print(f"tenant A reads B's discriminator-less source: {caveat_visible} of {len(b_papers)} "
      f"B rows visible")
print("the engine does not authenticate — access-gating lives above it")

This is the claim that is easy to overstate and that this chapter must not re-break: a
*separate per-tenant source* is hidden from the **listing** (Property 1), but its rows are
**not** hidden from a direct named read. Data isolation is the discriminator column
(Property 2), not source separation.

In [ ]:
assert listing_leak == 0 and discriminator_leak == 0  # the two hard zeros
contracts.assert_close("tenancy_b.caveat_visible", caveat_visible)

## Property 4 — the same recipe, scoped per tenant

The consequence worth measuring directly: the **same recipe** run under each tenant
gives that tenant its own answer, over only its own rows. The recipe is the book's
retrieval measurement — embed the papers (tier 01), build the same-subject golden, and
score it with `eval_embeddings` — run once inside each tenant's scope. Each embedding
table is built from, and each search reads, only the rows the discriminator lets that
tenant see; an unscoped session sees none of them.

In [ ]:
def scoped_precision(tenant_id: str) -> tuple[float, int, int]:
    with tenant(db, tenant_id):
        rows = db.sql(
            "SELECT paper_id, title, subject FROM papers_tagged.public.papers_tagged"
        ).to_pylist()
        embedded = db.generate_embeddings(
            source="papers_tagged", model=encoders.text(SCALE),
            columns=["title", "abstract"], key="paper_id",
        )
        golden = datasets.same_label_golden(
            db, rows, key="paper_id", label="subject", text="title", queries=100,
            name=f"golden_{tenant_id[:8]}",
        )
        report = db.eval_embeddings(
            source="papers_tagged", golden_source=golden, embedding_table=embedded, k=10,
        )
    return report["aggregate"]["precision_at_k"], len(rows), len(report["per_query"])


a_precision, a_visible, a_queries = scoped_precision(TENANT_A)
b_precision, b_visible, b_queries = scoped_precision(TENANT_B)
print(f"tenant A: precision@10 {a_precision:.3f}  over {a_visible} rows / {a_queries} queries")
print(f"tenant B: precision@10 {b_precision:.3f}  over {b_visible} rows / {b_queries} queries")
print(f"scopes tile the corpus: {a_visible} + {b_visible} = {a_visible + b_visible}")

In [ ]:
contracts.assert_close("tenancy_b.parity_a_precision_at_10", a_precision, tol=0.03)
contracts.assert_close("tenancy_b.parity_b_precision_at_10", b_precision, tol=0.03)
assert a_visible + b_visible == len(subject), "the tenant scopes tile the corpus"

The two numbers differ because the tenants hold different subjects, and some subjects
are easier to retrieve than others — but the point is the **parity**: one recipe, the
engine scoping the data per tenant, each tenant getting its own measured answer over rows
it alone can see.

The session's work is done, so it is closed. An embedded engine holds its catalog until
`close()` returns, which is why `close()` comes before anything removes the directory the
catalog lives in.

In [ ]:
db.close()

## What this chapter does not claim

The engine isolates at the catalog listing and at a discriminator-column scan rewrite.
It does **not** authenticate, and a discriminator-less source is globally readable — the
caveat is measured (Property 3), not glossed. Tenant isolation as a complete access-control
story is built *above* the engine (a discriminator column on every tenant-scoped table, or
a Flight SQL / gRPC interceptor that binds the scope from an authenticated principal). What
the engine guarantees — and what this chapter measures as hard zeros and a positive caveat
count — is exactly those two layers and that one honest limit.